In [1]:
from pathlib import Path
import pandas as pd, numpy as np

RAW = Path("../data/raw")   # change to ../data/raw/original if you nested it
frames = []
for f in RAW.rglob("*.csv"):
    name = f.stem.lower()
    sensor, room = name.split("_room_")
    df = pd.read_csv(f)
    df = df.rename(columns={f"room_{room}": "value"})
    df["sensor"], df["room"] = sensor, int(room)
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
raw["t"] = pd.to_timedelta(raw["Time"])
print(raw.shape)
raw.groupby(["sensor", "room"]).size().unstack()

(1087315, 9)


room,1,2,3
sensor,,,
co2,12044,11897,12059
humidity,6440,7639,5468
illuminance,24813,23663,21542
occupant_count,63350,63350,63350
occupant_count_cam1_bw,63334,63334,63288
occupant_count_cam1_fw,63334,63334,63288
occupant_count_cam2_bw,63170,63287,63335
occupant_count_cam2_fw,63170,63287,63335
temperature,5200,2457,2352


In [2]:
# how many distinct days, and does each DayId map to one Year/Month?
print("unique DayIds:", raw["DayId"].nunique())
print(raw.groupby("DayId")[["Year", "Month"]].nunique().max())

# do the rows within a day go in time order?
occ = raw[(raw.sensor == "occupant_count") & (raw.room == 1)]
print("time ordered within day:",
      occ.groupby("DayId")["t"].apply(lambda s: s.is_monotonic_increasing).all())

# how many days per month, and rows per day
print(occ.groupby(["Year", "Month"])["DayId"].nunique())
print(occ.groupby("DayId").size().describe())

unique DayIds: 44
Year     1
Month    1
dtype: int64
time ordered within day: True
Year  Month
2018  3         3
      5         5
      6         5
      7         1
      9         4
      10       10
      11        7
      12        3
2019  1         2
      2         4
Name: DayId, dtype: int64
count      44.000000
mean     1439.772727
std         0.423915
min      1439.000000
25%      1440.000000
50%      1440.000000
75%      1440.000000
max      1440.000000
dtype: float64


In [3]:
cov = (raw.groupby(["sensor", "room", "DayId"]).size()
          .unstack("DayId").fillna(0))
cov.astype(int).to_csv("../reports/coverage_by_day.csv")
(cov > 0).sum(axis=1).unstack(0)   # number of days with any data, per sensor/room

sensor,co2,humidity,illuminance,occupant_count,occupant_count_cam1_bw,occupant_count_cam1_fw,occupant_count_cam2_bw,occupant_count_cam2_fw,temperature,vav
room,,,,,,,,,,
1,44,44,44,44,44,44,44,44,44,44
2,44,44,44,44,44,44,44,44,44,44
3,44,44,44,44,44,44,44,44,44,44
